# Notebook 2: Generative AI Concepts for Physicists

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This notebook builds intuition for:
- What generative models do (and don't do)
- How **diffusion models** work — with physics analogies
- Why diffusion models are well-suited for crystal structure generation
- How **structural constraints** fit into the generation process

> This is mostly a conceptual notebook with visualizations. Minimal coding required.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
# This notebook only needs numpy, matplotlib, scipy (pre-installed on Colab).
# Install scipy just in case:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])
print('Ready.')

---
## 2.1 Discriminative vs. generative models

| | Discriminative | Generative |
|---|---|---|
| **Task** | Predict a label from data | Create new data samples |
| **Learns** | P(label \| data) | P(data) |
| **Materials example** | "Is this structure stable?" | "Generate a new stable structure" |

A generative model learns the **probability distribution** over its training data,
then samples from that distribution to create new examples.

**Physics analogy:** Think of the training data as defining an energy landscape.
The generative model learns this landscape, and sampling is like finding new
low-energy configurations.

---
## 2.2 A brief taxonomy of generative models

| Model | Key idea | Physics analogy |
|-------|----------|-----------------|
| **VAE** | Encode → latent space → decode | Compressed representation of configuration space |
| **GAN** | Generator vs. discriminator game | Adversarial optimization |
| **Flow** | Invertible transformations | Canonical transformations in phase space |
| **Diffusion** | Gradually add then remove noise | Heating → controlled cooling (annealing) |

**Diffusion models** have become the dominant approach for crystal generation
because they naturally handle the periodicity and symmetry of crystal structures.

---
## 2.3 How diffusion models work

### The forward process: adding noise (DDPM)

A **Denoising Diffusion Probabilistic Model** (DDPM) defines a Markov chain that
gradually corrupts data $\mathbf{x}_0$ into pure Gaussian noise over $T$ steps.

At each step $t$, a small amount of noise is added:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t;\, \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I})$$

where $\beta_t \in (0, 1)$ is the **noise schedule** — a small number that controls
how much noise is added at step $t$.

Thanks to the reparameterization trick, we can jump directly to any step $t$ without
iterating:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t;\, \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ is the
cumulative signal retention. As $t \to T$, $\bar{\alpha}_T \approx 0$ and the data
becomes pure noise.

**Physics analogy:** This is like **heating a crystal until it melts** into a
disordered liquid. $\beta_t$ is the heating rate, and $\bar{\alpha}_t$ tracks
how much crystalline order remains.

### The reverse process: denoising

The reverse process learns to undo the noising, generating data from noise:

$$p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1};\, \boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\; \sigma_t^2 \mathbf{I})$$

A neural network $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ is trained to predict the noise
$\boldsymbol{\epsilon}$ that was added, minimizing the simple loss:

$$\mathcal{L} = \mathbb{E}_{t, \mathbf{x}_0, \boldsymbol{\epsilon}}\left[\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\|^2\right]$$

Once trained, the predicted mean is:

$$\boldsymbol{\mu}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\right)$$

**Physics analogy:** This is like **controlled crystallization from a melt**.
The model has learned the "crystallization dynamics" and guides random atoms
into a coherent crystal.

### The score function perspective

Equivalently, the network learns the **score function** $\nabla_{\mathbf{x}} \log p(\mathbf{x})$ —
the gradient of the log-probability density. In plain terms: "which direction should
I move each atom to make this look more like a real crystal?"

The noise prediction $\boldsymbol{\epsilon}_\theta$ and score $\mathbf{s}_\theta$ are related by:

$$\mathbf{s}_\theta(\mathbf{x}_t, t) = -\frac{\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)}{\sqrt{1 - \bar{\alpha}_t}}$$

### For crystal structures (SCIGEN)

In SCIGEN, the diffusion process operates simultaneously on three components:
- **Atom positions** $\mathbf{F}$ (fractional coordinates, with wrapping for periodicity)
- **Atom types** $\mathbf{A}$ (element species — discrete diffusion)
- **Lattice matrix** $\mathbf{L}$ (the unit cell shape — 3×3 matrix)

The combined loss is: $\mathcal{L} = \mathcal{L}_\text{coords} + \mathcal{L}_\text{types} + \mathcal{L}_\text{lattice}$

Let's visualize these concepts with a simple 2D example.

### 2.4 Toy demo: the forward process

The plot below shows data points arranged in a ring of 6 clusters (like atoms at
lattice sites). As we increase the noise level $\sigma$, the points diffuse outward
until the original structure is lost — analogous to melting a crystal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

np.random.seed(42)

# Define a simple 2D target distribution (mixture of Gaussians arranged in a ring)
def target_samples(n=500):
    """Generate samples from a ring of Gaussians (like crystal sites)."""
    n_centers = 6
    angles = np.linspace(0, 2*np.pi, n_centers, endpoint=False)
    centers = np.stack([2*np.cos(angles), 2*np.sin(angles)], axis=1)
    samples = []
    for _ in range(n):
        c = centers[np.random.randint(n_centers)]
        samples.append(c + 0.15 * np.random.randn(2))
    return np.array(samples)

# Simulate the forward (noising) process
data = target_samples(500)
noise_levels = [0.0, 0.3, 0.8, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sigma in zip(axes, noise_levels):
    noised = data + sigma * np.random.randn(*data.shape)
    ax.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.5, c='steelblue')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    if sigma == 0:
        ax.set_title('Original data\n(crystal sites)', fontsize=11)
    else:
        ax.set_title(f'Noise level σ = {sigma}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Forward process: gradually adding noise ("heating")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Toy demo: the reverse (denoising) process

Now we simulate the **reverse process**: starting from pure noise, we iteratively
denoise toward the target distribution. Our toy denoiser estimates the score function
by finding the direction toward the nearest data clusters:

$$\hat{\mathbf{s}}(\mathbf{x}) \approx \frac{1}{k}\sum_{i \in \text{kNN}(\mathbf{x})} (\mathbf{x}_i - \mathbf{x})$$

At each step, we move points in the score direction and add a small amount of noise
(the stochastic part of Langevin dynamics):

$$\mathbf{x}_{t-1} = \mathbf{x}_t + \eta\,\hat{\mathbf{s}}(\mathbf{x}_t) + \sqrt{2\eta}\,\boldsymbol{\xi}, \quad \boldsymbol{\xi} \sim \mathcal{N}(0, \mathbf{I})$$

Watch how the disordered "melt" gradually crystallizes back into the ring pattern.

In [ ]:
# Simulate the reverse (denoising) process
# Starting from noise, gradually denoise toward the target distribution

def simple_denoise_step(x, data, sigma_current, sigma_next, n_neighbors=10):
    """A toy denoising step: move points toward nearest cluster centers."""
    from scipy.spatial.distance import cdist
    # Estimate score: direction toward nearest data points
    dists = cdist(x, data)
    nearest_idx = np.argsort(dists, axis=1)[:, :n_neighbors]
    target = np.mean(data[nearest_idx], axis=1)
    # Move partway toward target
    step_size = 1 - sigma_next / sigma_current
    x_new = x + step_size * (target - x)
    # Add small noise (except at final step)
    if sigma_next > 0:
        x_new += sigma_next * 0.3 * np.random.randn(*x.shape)
    return x_new

# Start from pure noise
x = 3.5 * np.random.randn(500, 2)
sigmas = [2.0, 0.8, 0.3, 0.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c='coral')
axes[0].set_title('Pure noise\n("disordered melt")', fontsize=11)
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5)
axes[0].set_aspect('equal'); axes[0].set_xticks([]); axes[0].set_yticks([])

for i in range(1, 4):
    x = simple_denoise_step(x, data, sigmas[i-1], sigmas[i] if i < 3 else 0)
    color = ['coral', 'mediumpurple', 'steelblue'][i-1]
    axes[i].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c=color)
    if i < 3:
        axes[i].set_title(f'Denoising step {i}', fontsize=11)
    else:
        axes[i].set_title('Generated samples\n("crystallized")', fontsize=11)
    axes[i].set_xlim(-5, 5); axes[i].set_ylim(-5, 5)
    axes[i].set_aspect('equal'); axes[i].set_xticks([]); axes[i].set_yticks([])

fig.suptitle('Reverse process: denoising ("controlled crystallization")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Animated view

Here's the same forward → reverse process as an animation:

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig_anim, ax_anim = plt.subplots(figsize=(5, 5))

# Forward then reverse: 20 frames
n_frames = 20
sigmas_fwd = np.linspace(0, 2.5, n_frames // 2)
sigmas_rev = sigmas_fwd[::-1]
all_sigmas = np.concatenate([sigmas_fwd, sigmas_rev])

def animate(frame):
    ax_anim.clear()
    sigma = all_sigmas[frame]
    if frame < n_frames // 2:
        noised = data + sigma * np.random.randn(*data.shape)
        color = 'steelblue'
        phase = f'Forward (σ={sigma:.1f})'
    else:
        # Simple denoising approximation
        noise_level = sigma
        noised = data + noise_level * np.random.randn(*data.shape)
        color = '#E69F00'
        phase = f'Reverse (σ={sigma:.1f})'
    ax_anim.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.5, c=color)
    ax_anim.set_xlim(-5, 5)
    ax_anim.set_ylim(-5, 5)
    ax_anim.set_aspect('equal')
    ax_anim.set_xticks([])
    ax_anim.set_yticks([])
    ax_anim.set_title(phase, fontsize=12)

anim = FuncAnimation(fig_anim, animate, frames=n_frames, interval=300)
plt.close(fig_anim)
HTML(anim.to_jshtml())

### The noise schedule

The noise schedule $\{\beta_t\}_{t=1}^T$ controls the rate of corruption. A **linear
schedule** starts with small $\beta_1 \approx 10^{-4}$ and increases to $\beta_T \approx 0.02$.

The cumulative product $\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$ tells us how
much of the original signal survives at step $t$. When $\bar{\alpha}_t \approx 0$,
the data is essentially pure noise.

In [ ]:
# Noise schedule visualization
T = 1000  # total diffusion steps
betas = np.linspace(1e-4, 0.02, T)  # linear schedule
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(T), betas, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Diffusion step t')
axes[0].set_ylabel('β_t (noise added per step)')
axes[0].set_title('Noise schedule β_t')

axes[1].plot(range(T), alpha_bars, color='#E69F00', linewidth=1.5)
axes[1].set_xlabel('Diffusion step t')
axes[1].set_ylabel('ᾱ_t (signal remaining)')
axes[1].set_title('Cumulative signal retention ᾱ_t')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% signal')
axes[1].legend()

plt.tight_layout()
plt.show()

print('At t=0: structure is clean (ᾱ ≈ 1)')
print(f'At t={T}: structure is nearly pure noise (ᾱ = {alpha_bars[-1]:.6f})')

The toy example above shows the core idea:
- The **forward process** destroys structure by adding noise
- The **reverse process** creates structure by removing noise
- The neural network learns to perform the reverse process

In a real crystal diffusion model (like SCIGEN), the same principle applies
but in 3D, with periodic boundary conditions, and the "data" includes:
- Atom positions (fractional coordinates)
- Atom types (element species)
- Lattice parameters (the unit cell shape)

---
## 2.5 Conditional generation and structural constraints

A diffusion model can generate **unconditionally** (any valid crystal) or
**conditionally** (crystals with specific properties):

| Conditioning type | Example | Method |
|---|---|---|
| Property-guided | "Band gap > 2 eV" | Classifier guidance |
| Composition-guided | "Must contain Mn and O" | Fix atom types during denoising |
| **Structure-guided** | **"Must have kagome sublattice"** | **Inpainting with constrained atoms** |

### SCIGEN's inpainting approach

SCIGEN uses **structure-guided generation** analogous to image inpainting — filling
in missing parts of a picture. The key idea:

Partition the atoms into **known** (constrained) and **unknown** (generated):

$$\mathbf{x} = (\mathbf{x}^\text{known},\; \mathbf{x}^\text{unknown})$$

During denoising, only $\mathbf{x}^\text{unknown}$ follows the learned reverse
process. The known atoms $\mathbf{x}^\text{known}$ (defining the kagome/honeycomb
pattern) are **revealed at the final step**, replacing whatever the model generated
at those sites:

$$\mathbf{x}_0^\text{known} = \mathbf{x}_\text{constraint}$$

This is different from clamping atoms at every step. Instead, the model freely
denoises all positions, then the constraint is applied once at the end. During
training, the model sees the constrained positions and learns to **anticipate**
them — generating complementary atoms that form a physically reasonable crystal
around the fixed sublattice.

The lattice geometry is also constrained (e.g., hexagonal cell with $\gamma = 120°$
for kagome), ensuring the generated structure is compatible with the desired pattern.

In [ ]:
# Visualize the inpainting idea: constrained kagome sites + generated additional atoms
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Kagome constraint points
a = 5.0
gamma = np.radians(120)
lat_a = np.array([a, 0])
lat_b = np.array([a * np.cos(gamma), a * np.sin(gamma)])

kagome_frac = np.array([[0.5, 0.0], [0.0, 0.5], [0.5, 0.5]])

def frac_to_cart(frac, lat_a, lat_b):
    return frac[:, 0:1] * lat_a + frac[:, 1:2] * lat_b

# Plot 1: The desired constraint pattern
ax = axes[0]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Target: kagome sublattice\n(will be "inpainted")', fontsize=11)

# Plot 2: Model generates from noise (all atoms are noisy)
ax = axes[1]
np.random.seed(7)
all_noisy = np.random.uniform(-1, 11, (45, 2))
ax.scatter(all_noisy[:, 0], all_noisy[:, 1], s=60, c='lightgray',
           edgecolors='gray', linewidths=0.5, alpha=0.7, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Denoising: model generates\nall atoms from noise', fontsize=11)

# Plot 3: After denoising, kagome sites are revealed (inpainted)
ax = axes[2]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)

# Simulated generated additional atoms at sensible positions
extra_frac = np.array([[0.33, 0.33], [0.17, 0.17], [0.67, 0.33], [0.33, 0.67]])
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = extra_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=60, c='steelblue',
                   edgecolors='black', linewidths=0.5, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Result: kagome sites inpainted\n+ generated atoms (blue)', fontsize=11)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## Exercise

1. Modify the toy 2D code: change `n_centers = 6` to `n_centers = 4` (square pattern instead of hexagonal). Run the forward and reverse process. Does the denoising still recover the pattern?
2. What happens if you increase the maximum noise level from 2.0 to 5.0? Does the reverse process still work?

In [ ]:
# Your code here

---
## 2.6 The landscape of crystal generative models

| Model | Year | Type | Key feature |
|-------|------|------|-------------|
| CDVAE | 2022 | VAE | First crystal generative model with graph decoder |
| DiffCSP | 2023 | Diffusion | Crystal structure prediction from composition |
| **SCIGEN** | **2025** | **Diffusion** | **Structural constraints (kagome, honeycomb, etc.)** |
| MatterGen | 2025 | Diffusion | Property-guided generation |
| UniMat | 2025 | Diffusion | Universal materials generation |

What makes SCIGEN unique: it is the first model that lets you specify **geometric
constraints** on the generated structure. Instead of just saying "generate something
stable," you can say "generate a structure with a kagome sublattice" — and the model
will fill in the rest of the crystal around that constraint.

---
## 2.7 Available structural constraints in SCIGEN

SCIGEN supports 13 lattice types. Here are the most commonly used:

The image below shows the geometric patterns for each constraint type:

![Structural constraint lattice types](../assets/SI_arch_lattice_unit_bk.png)

In [ ]:
# Available structural constraints in SCIGEN
sc_table = {
    'tri':  ('Triangular',              1, '60°'),
    'hon':  ('Honeycomb',               2, '120°'),
    'kag':  ('Kagome',                  3, '120°'),
    'sqr':  ('Square',                  1, '90°'),
    'lieb': ('Lieb',                    3, '90°'),
    'elt':  ('Elongated triangular',    2, 'variable'),
    'sns':  ('Snub square',             4, 'variable'),
    'tsq':  ('Truncated square',        4, '90°'),
    'srt':  ('Small rhombotrihexagonal', 6, '120°'),
    'snh':  ('Snub hexagonal',          6, '120°'),
    'trh':  ('Truncated hexagonal',     6, '120°'),
    'grt':  ('Great rhombotrihexagonal', 12, '120°'),
    'van':  ('Vanilla (no constraint)',  1, 'any'),
}

print(f'{"Code":<6} {"Lattice type":<30} {"Known atoms":<14} {"γ angle"}')
print('-' * 60)
for code, (name, n_known, gamma) in sc_table.items():
    print(f'{code:<6} {name:<30} {n_known:<14} {gamma}')

---
## Key takeaways

1. **Diffusion models** (DDPM) generate crystals by learning to reverse a noising process — like controlled crystallization from a melt.
2. The forward process adds Gaussian noise: $q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$
3. The reverse process learns a denoiser $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ that predicts the noise to remove at each step.
4. **SCIGEN** adds structural constraints via inpainting: fix some atoms in a desired geometric pattern (kagome, honeycomb, etc.), let the model generate the rest.
5. This enables **targeted discovery** of materials with specific lattice geometries — precisely the rare structures that are hard to find by screening existing databases.

In the next notebook, we'll learn how to **evaluate** whether generated structures are physically reasonable.

---
## References

- **DDPM:** Ho, Jain & Abbeel, "Denoising Diffusion Probabilistic Models," *NeurIPS* (2020). [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
- **Score matching:** Song & Ermon, "Generative Modeling by Estimating Gradients of the Data Distribution," *NeurIPS* (2019). [arXiv:1907.05600](https://arxiv.org/abs/1907.05600)
- **CDVAE:** Xie et al., "Crystal Diffusion Variational Autoencoder for Periodic Material Generation," *ICLR* (2022). [arXiv:2110.06197](https://arxiv.org/abs/2110.06197)
- **DiffCSP:** Jiao et al., "Crystal Structure Prediction by Joint Equivariant Diffusion," *NeurIPS* (2023). [arXiv:2309.04475](https://arxiv.org/abs/2309.04475)
- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y)
- **MatterGen:** Zeni et al., "MatterGen: a generative model for inorganic materials design," *Nature* (2025). [DOI](https://doi.org/10.1038/s41586-025-08628-5)

---
## What's next?

Proceed to **Notebook 03: DiffCSP Generated Materials** to learn about how noise and denoising work for crystal structures.